# Contoso PMO KB: agent queries

Sends representative questions to the **contoso-pmo-agent** created in [`08-05-01-contoso-pmo-agent-setup.ipynb`](08-05-01-contoso-pmo-agent-setup.ipynb). Run the setup notebook first to deploy the Azure Functions MCP server and create the agent on the admin project, then use this notebook to query it interactively.

```
contoso-pmo-agent  ←── Foundry agent (37 MCP tools)
        │           on project-admin-{suffix}
        ▼ Responses API (openai_client.responses.create)
   This notebook
```

## Prerequisites

1. **Agent exists** - complete [`08-05-01-contoso-pmo-agent-setup.ipynb`](08-05-01-contoso-pmo-agent-setup.ipynb) through Phase 5 so `contoso-pmo-agent` exists on `project-admin-{suffix}`.
2. **Python environment** - run `uv sync` from the repo root; select the `.venv` kernel.
3. **`.env`** - `CHAT_MODEL` only (admin endpoint is derived deterministically from the subscription ID).
4. **Azure CLI** - `az login` (required by `DefaultAzureCredential`).

## Imports and configuration

In [1]:
import hashlib
import os
import subprocess
from pathlib import Path

from dotenv import load_dotenv

repo_root = Path(
    subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip()
)
load_dotenv(repo_root / '.env', override=True)

CHAT_MODEL = os.environ.get('CHAT_MODEL', 'gpt-4.1-mini')

SUBSCRIPTION_ID = (
    os.environ.get('AZURE_SUBSCRIPTION_ID')
    or subprocess.run('az account show --query id -o tsv',
                      shell=True, capture_output=True, text=True).stdout.strip()
)
SUFFIX           = hashlib.sha256((SUBSCRIPTION_ID + 'v2').encode()).hexdigest()[:6]
PROJECT_ENDPOINT = f'https://aif-core-{SUFFIX}.services.ai.azure.com/api/projects/project-admin-{SUFFIX}'

AGENT_NAME = 'contoso-pmo-agent'

print(f'Project endpoint : {PROJECT_ENDPOINT}')
print(f'Chat model       : {CHAT_MODEL}')
print(f'Agent name       : {AGENT_NAME}')

Project endpoint : https://aif-core-c2676f.services.ai.azure.com/api/projects/project-admin-c2676f
Chat model       : gpt-4.1-mini
Agent name       : contoso-pmo-agent


## Authenticate and connect

In [2]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai_client  = project_client.get_openai_client()
print('AIProjectClient + Responses-API openai_client initialised.')


AIProjectClient + Responses-API openai_client initialised.


## Resolve the agent and set up the ask() helper

Pulls the latest version of `contoso-pmo-agent` (created in [`08-05-01-contoso-pmo-agent-setup.ipynb`](08-05-01-contoso-pmo-agent-setup.ipynb)) and defines an `ask()` helper that invokes the agent via the Responses API (`openai_client.responses.create` with an `agent_reference` extra body). `require_approval='never'` is baked into the agent's tool definition, so no per-run approval plumbing is needed.


In [3]:
# Pull the latest version of the agent (versions are append-only on the new API).
versions = list(project_client.agents.list_versions(agent_name=AGENT_NAME))
if not versions:
    raise RuntimeError(
        f"Agent '{AGENT_NAME}' not found on {PROJECT_ENDPOINT}. "
        f"Run 08-05-01-contoso-pmo-agent-setup.ipynb first."
    )
agent = versions[0]
print(f"Using agent: {agent.name} v{agent.version}")

AGENT_REF = {'agent_reference': {'name': agent.name, 'version': agent.version, 'type': 'agent_reference'}}


def ask(question: str) -> str:
    """Send a question to the Contoso PMO KB agent and return the assistant's response text.

    Uses the Responses API surface - `require_approval='never'` is baked into the
    agent's tool definition (see Phase 5 in 08-05-01), so no per-run approval
    plumbing is needed here. Raises if the run does not complete cleanly so
    failures don't silently surface as empty output.
    """
    response = openai_client.responses.create(
        input=[{'role': 'user', 'content': question}],
        extra_body=AGENT_REF,
    )
    if response.status != 'completed' or response.error:
        raise RuntimeError(
            f"Agent run did not complete cleanly.\n"
            f"  status     : {response.status}\n"
            f"  error      : {response.error}\n"
            f"  response_id: {response.id}"
        )
    text = response.output_text
    if not text:
        raise RuntimeError(
            f"Run completed but produced no text output (response_id={response.id})."
        )
    return text


print('ask() ready. Run the query cells below.')


Using agent: contoso-pmo-agent v1
ask() ready. Run the query cells below.


---
## Query the agent

Sends questions through the agent via the Responses API. Each `ask()` call can trigger many sequential MCP tool calls, each consuming tokens against the `CHAT_MODEL` deployment on the hub (`aif-core-{suffix}`).

> **Rate limits:** if you hit a 429, increase the TPM quota for `CHAT_MODEL` on the hub account in the Azure AI Foundry portal.


In [4]:
print(ask(
    "What tasks are currently overdue across all projects? "
    "Who is responsible for each, and which project do they belong to, and what is the due date?"
))

Here are the currently overdue tasks across all projects:

1. Task: Prepare G3 gate update slide for governance board presentation
   - Responsible: Jane Smith (Project Manager, OCM)
   - Project: Project Aurora
   - Due Date: 2026-02-24

2. Task: Submit G5 regulatory compliance documentation for France market
   - Responsible: Tom Walsh (Team Member, Quality & Compliance)
   - Project: Project Bolt
   - Due Date: 2026-02-24

3. Task: Finalise consumer launch campaign brief and share with Categories for sign-off
   - Responsible: Carlos Mendez (Stakeholder, Marketing)
   - Project: Project Bolt
   - Due Date: 2026-02-25

Would you like me to assist with anything specific regarding these tasks?


In [5]:
print(ask(
    "I'm planning the Aurora G4 gate review. "
    "Search for relevant lessons learned from past projects and identify any risk "
    "patterns around supplier or regulatory issues that I should be aware of."
))

I searched for lessons learned and risk patterns related to supplier and regulatory issues specifically for the G4 gate, but no relevant items were found from past projects. If you want, I can help you identify potential risks to flag or areas to focus on based on other knowledge or project details. Would you like me to do that?


In [6]:
print(ask(
    "Search for documents containing discussion of scope changes or variant additions. "
    "Summarise the key decisions made and who was involved."
))

Here is a summary of key decisions and involved parties regarding scope changes or variant additions found in documents:

1. Project Aurora - G3 Gate Review (MoM doc-001, date 2026-02-20)
- Decision: G3 gate conditionally passed with a revised fill weight from 240ml to 250ml to meet cost targets.
- Fragrance variant scope was deferred to G4 pending commercial assessment by Categories.
- Packaging dieline to be updated to reflect revised fill weight before G4.
- Involved: Jane Smith (PM), Marco Rossi (R&D), Sarah Chen (Procurement), Tom Walsh (Quality), David Osei (Categories), Lisa Park (Packaging).

2. Project Aurora - Weekly Standup W8 (MoM doc-002, date 2026-02-26)
- Decision: Categories requested to add fragrance variant back into scope for UK market only.
- Impact assessment required from R&D and Packaging teams.
- Jane Smith to update and circulate the project plan with this scope change by 1 March.
- Scope review meeting scheduled with Categories and Packaging.
- Involved: Jane 